# QuantStream — tự train TCN bằng PyTorch trên Kaggle

Upload `QuantStream_Kaggle_Input.zip` thành Dataset riêng tư, Add Input vào notebook. Bật Internet và chọn GPU nếu tài khoản còn hạn mức. Mã và snapshot được lấy từ ZIP; final holdout không được xuất sang Kaggle.

Notebook train ba fold walk-forward, chọn epoch/threshold bằng development, refit model cuối trên development rồi xuất weights và metadata. Đây là train từ đầu, không phải fine-tune pretrained.

In [ ]:
import subprocess,sys
# Chỉ cài dependencies cần cho training; không cài dashboard trong notebook.
subprocess.check_call([sys.executable,'-m','pip','install','-q','pyarrow>=17,<27','scikit-learn>=1.5,<2'])

In [ ]:
from pathlib import Path
import sys, zipfile, shutil, json

WORK = Path('/kaggle/working/quantstream') if Path('/kaggle/working').exists() else Path.cwd()/'notebook_work'
WORK.mkdir(parents=True,exist_ok=True)
INPUT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd().parent
archives = list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            p=Path(name)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('Invalid archive path')
        z.extractall(WORK)
else:
    # Kaggle may automatically extract a ZIP uploaded as a Dataset.
    manifests=[p for p in INPUT.rglob('manifest.json') if (p.parent/'development.parquet').exists()]
    if len(manifests)!=1:
        raise FileNotFoundError('Attach the Dataset containing QuantStream_Kaggle_Input.zip (or its extracted contents).')
    source=manifests[0].parent
    shutil.copytree(source,WORK/'snapshot',dirs_exist_ok=True)
    shutil.copytree(source.parent/'code',WORK/'code',dirs_exist_ok=True)
SNAPSHOT=WORK/'snapshot'
sys.path.insert(0,str(WORK/'code'))
manifest=json.loads((SNAPSHOT/'manifest.json').read_text())
assert manifest['contains_final_holdout'] is False
print('Snapshot:',SNAPSHOT)
print('Final holdout included:',manifest['contains_final_holdout'])

In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))

## Train

TCN dùng convolution causal có dilation; không nhìn các bước phía sau. Scaler của mỗi fold chỉ fit train. Huber loss trên lợi suất được nhân 100 để ổn định thang số. Chọn epoch bằng validation MSE. Strategy threshold được so sánh trên ba fold sau phí.

`epochs=8` là cấu hình dự kiến cho lần chạy Kaggle. Lượt kiểm chứng kèm gói đã dùng 6 epoch/fold và lưu kết quả riêng.

In [ ]:
from quantstream.deep import train_snapshot
OUTPUT=WORK/'tcn_output'
metadata=train_snapshot(SNAPSHOT,OUTPUT,epochs=8,device='auto')
print(json.dumps(metadata['selection'],indent=2))

## Download output

Download ZIP trong Output, giải nén về local rồi xem báo cáo validation trước khi chọn model. `python -m quantstream register-tcn <folder>` kiểm tra hash và đăng ký model. Holdout đã mở từ trước vẫn bị khóa với model mới; dùng giai đoạn tương lai mới hoặc forward paper trading.

In [ ]:
archive=shutil.make_archive(str(WORK.parent/'TCN_Kaggle_Output'),'zip',OUTPUT)
print('Download:',archive)
from IPython.display import FileLink,display
display(FileLink(archive))